In [17]:
import os
import pandas as pd
import numpy as np

# Get the directory where notebook is currently located
base_path = os.getcwd() 

# Combine directory with your data folder and file name
file_path = os.path.join(base_path,'Health Care_Patient_survey_source.csv')
df_raw = pd.read_csv(file_path)
#Read CSV_File
df = df_raw.copy()

print("Raw Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())

Raw Shape: (34999, 23)
Columns: ['Provider ID', 'Hospital Name', 'Address', 'City', 'State', 'ZIP Code', 'County Name', 'Phone Number', 'Measure ID', 'Question', 'Answer Description', 'Patient Survey Star Rating', 'Patient Survey Star Rating Footnote', 'Answer Percent', 'Answer Percent Footnote', 'Linear Mean Value', 'Number of Completed Surveys', 'Number of Completed Surveys Footnote', 'Survey Response Rate Percent', 'Survey Response Rate Percent Footnote', 'Measure Start Date', 'Measure End Date', 'Location']


In [18]:
# ---------------------------------------------------------
# Adding '_' to column names
# ---------------------------------------------------------
df.columns = [col.replace(' ', '_') for col in df.columns]

print("Copy Shape:", df.shape)
print("Columns:", df.columns.tolist())

Copy Shape: (34999, 23)
Columns: ['Provider_ID', 'Hospital_Name', 'Address', 'City', 'State', 'ZIP_Code', 'County_Name', 'Phone_Number', 'Measure_ID', 'Question', 'Answer_Description', 'Patient_Survey_Star_Rating', 'Patient_Survey_Star_Rating_Footnote', 'Answer_Percent', 'Answer_Percent_Footnote', 'Linear_Mean_Value', 'Number_of_Completed_Surveys', 'Number_of_Completed_Surveys_Footnote', 'Survey_Response_Rate_Percent', 'Survey_Response_Rate_Percent_Footnote', 'Measure_Start_Date', 'Measure_End_Date', 'Location']


In [69]:
#-----------------------------------
# 🟢 Structural Validation
#------------------------------------

df.info()
df.head()
df.describe(include="all")

In [19]:
#-------------------------------------------------
#🟢 Check for Duplicates
#-------------------------------------------------
df.duplicated().any()

False

In [20]:
#-------------------------------------
# 🟢 Standardize NULL Values
#-------------------------------------
NULL_VALUES = {
    "", " ", "nan", "none", "null", "n/a", "na",
    "unknown", "Unknown", "N/A", "Not Available"
}

def clean_null(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return np.nan if x in NULL_VALUES else x  #checks if a value is missing or matches a predefined list of null-like strings and returns NaN

for col in df.columns:
    df[col] = df[col].apply(clean_null) #applies the clean_null function to every value in each column of the DataFrame, replacing each column with its cleaned version.

df.isna().mean().sort_values(ascending=False) #calculates the proportion of missing values in each column

Patient_Survey_Star_Rating_Footnote      0.950341
Answer_Percent_Footnote                  0.852510
Survey_Response_Rate_Percent_Footnote    0.745707
Number_of_Completed_Surveys_Footnote     0.745707
Survey_Response_Rate_Percent             0.115718
Number_of_Completed_Surveys              0.115718
Answer_Percent                           0.067116
Patient_Survey_Star_Rating               0.049659
Linear_Mean_Value                        0.045144
County_Name                              0.007143
Measure_End_Date                         0.000000
Measure_Start_Date                       0.000000
Provider_ID                              0.000000
Hospital_Name                            0.000000
Answer_Description                       0.000000
Question                                 0.000000
Measure_ID                               0.000000
Phone_Number                             0.000000
ZIP_Code                                 0.000000
State                                    0.000000


In [21]:
#--------------------------------------------------------------------------------
# Checking columns that hold numeric values before Data Loss Audit is Performed
#--------------------------------------------------------------------------------
cols = [
        'Answer_Percent',
        'Patient_Survey_Star_Rating',
        'Linear_Mean_Value',
        'Number_of_Completed_Surveys',
        'Survey_Response_Rate_Percent'
    ]

def print_unique_samples(df, columns, n=20):
    for col in columns:
        print(f"{col}:")
        print(df[col].unique()[:n])
        print()

print_unique_samples(df, cols)

Answer_Percent:
['63' '13' '24' 'Not Applicable' '70' '9' '21' '78' '7' '15' '47' '19'
 '34' '53' '27' '20' '85' '46' '60' '67']

Patient_Survey_Star_Rating:
['Not Applicable' '2' '3' '1' '4' nan '5']

Linear_Mean_Value:
['Not Applicable' '83' '87' '90' '75' '70' '85' '79' '86' '84' '91' '94'
 '81' '88' '89' '76' '80' '82' nan '77']

Number_of_Completed_Surveys:
['506' '1135' '579' '185' '63' '2193' '513' '1103' '38' '578' '141' '92'
 '702' '1751' '2262' '49' '5024' '224' '417' '344']

Survey_Response_Rate_Percent:
['21' '34' '22' '27' '31' '33' '23' '41' '19' '25' '32' '24' '20' '17'
 '35' '30' nan '28' '37' '15']



In [22]:
#----------------------------------------------------
# Performing a Data Loss Audit on Numeric columns
#----------------------------------------------------

# Check non-nulls BEFORE conversion
numeric_cols = ['Answer_Percent', 'Linear_Mean_Value', 'Patient_Survey_Star_Rating', 
                'Number_of_Completed_Surveys', 'Survey_Response_Rate_Percent']
before_counts = df[numeric_cols].count()

# Perform the conversion
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Check non-nulls AFTER conversion
after_counts = df[numeric_cols].count()

# Calculate the "Silent Loss"
loss_report = before_counts - after_counts
print("--- Data Loss Audit ---")
print(loss_report[loss_report > 0]) # Only show columns where data was lost

--- Data Loss Audit ---
Answer_Percent                14699
Linear_Mean_Value             27999
Patient_Survey_Star_Rating    27300
dtype: int64


In [23]:
#-------------------------------
# 🟢 Enforce Data Types
#-------------------------------
#Convert columns to datetime format
df['Measure_Start_Date'] = pd.to_datetime(df['Measure_Start_Date'],dayfirst=False, errors='coerce')
df['Measure_End_Date'] = pd.to_datetime(df['Measure_End_Date'],dayfirst=False, errors='coerce')

#Optimize categorical data
cat_cols = ['State', 'Measure_ID', 'Answer_Description']
for col in cat_cols:
    df[col] = df[col].astype('category')

In [24]:
#-----------------------------------
# Business Rule Validation (BRV)
#-----------------------------------
def run_business_rule_validation(df):
    print("--- Executing Business Rule Validation ---")
    
    # Rule 1: Star Ratings must be between 1 and 5
    invalid_stars = df[(df['Patient_Survey_Star_Rating'].notna()) & 
                       ((df['Patient_Survey_Star_Rating'] < 1) | (df['Patient_Survey_Star_Rating'] > 5))]
    
    # Rule 2: Percentages must be between 0 and 100
    invalid_pct = df[(df['Answer_Percent'].notna()) & 
                     ((df['Answer_Percent'] < 0) | (df['Answer_Percent'] > 100))]
    
    # Rule 3: End Date must be after Start Date
    invalid_dates = df[df['Measure_End_Date'] <= df['Measure_Start_Date']]
    
    # Rule 4: Logic Check - Surveys completed vs Response Rate
    # (If surveys > 0, response rate shouldn't be 0 or NaN)
    logic_gap = df[(df['Number_of_Completed_Surveys'] > 0) & 
                   (df['Survey_Response_Rate_Percent'].isna() | (df['Survey_Response_Rate_Percent'] <= 0))]

    # Reporting the findings
    results = {
        "Invalid Star Ratings": len(invalid_stars),
        "Invalid Percentages": len(invalid_pct),
        "Chronology Errors (Dates)": len(invalid_dates),
        "Response Rate Logic Gaps": len(logic_gap)
    }
    
    for rule, count in results.items():
        status = "✅ PASS" if count == 0 else f"❌ FAIL ({count} records)"
        print(f"{rule}: {status}")
        
    return results

# Execute the validation
validation_results = run_business_rule_validation(df)

--- Executing Business Rule Validation ---
Invalid Star Ratings: ✅ PASS
Invalid Percentages: ✅ PASS
Chronology Errors (Dates): ✅ PASS
Response Rate Logic Gaps: ✅ PASS


In [25]:
#-------------------------------------------
# 🟢 Outlier Detection (Flag Only)
#-------------------------------------------
# Calculating IQR for Answer Percent to find the extremes
Q1 = df['Answer_Percent'].quantile(0.25)
Q3 = df['Answer_Percent'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Answer_Percent'] < lower_bound) | (df['Answer_Percent'] > upper_bound)]

print(f"Number of statistical outliers detected: {len(outliers)}")

Number of statistical outliers detected: 0


In [26]:
# The "Small Sample" Filter
low_volume_cutoff = 100
low_volume_count = len(df[df['Number_of_Completed_Surveys'] < low_volume_cutoff])

print(f"Records with < {low_volume_cutoff} surveys: {low_volume_count}")

Records with < 100 surveys: 3850


In [27]:
# Identify the bottom 5th percentile of hospitals

#Calculate the mean score per hospital across ALL categories
hospital_summary = df.groupby('Provider_ID')['Answer_Percent'].mean().reset_index()

#Find the 5th percentile of these hospital-wide averages
hospital_floor = hospital_summary['Answer_Percent'].quantile(0.05)

#Filter to find the unique hospitals below that floor
urgent_hospitals = hospital_summary[hospital_summary['Answer_Percent'] <= hospital_floor]

print(f"Hospital-wide Performance Floor: {hospital_floor:.2f}%")
print(f"Unique Hospitals in 'Urgent Improvement' zone: {len(urgent_hospitals)}")

Hospital-wide Performance Floor: 34.48%
Unique Hospitals in 'Urgent Improvement' zone: 619


In [28]:
# 1. Group by hospital and count HOW MANY scores they actually have
hospital_stats = df.groupby('Provider_ID')['Answer_Percent'].agg(['mean', 'count']).reset_index()

# 2. Filter out hospitals that have fewer than, say, 5 valid measures 
# (This ensures we aren't judging a hospital on a single noisy data point)
reliable_hospitals = hospital_stats[hospital_stats['count'] >= 5]

# 3. Now calculate the 5th percentile on the RELIABLE set
real_floor = reliable_hospitals['mean'].quantile(0.05)
urgent_hospitals = reliable_hospitals[reliable_hospitals['mean'] <= real_floor]

print(f"Total Unique Hospitals: {len(hospital_stats)}")
print(f"Reliable Hospitals (with >= 5 measures): {len(reliable_hospitals)}")
print(f"Corrected Performance Floor: {real_floor:.2f}%")
print(f"Unique Hospitals in 'Urgent Improvement' zone: {len(urgent_hospitals)}")

Total Unique Hospitals: 700
Reliable Hospitals (with >= 5 measures): 619
Corrected Performance Floor: 34.48%
Unique Hospitals in 'Urgent Improvement' zone: 619


In [29]:
# 1. Filter for 'Always' or 'Yes' descriptions (The "Top-Box" scores)
# You may need to check your unique 'Answer_Description' values to get the exact string
top_box_data = df[df['Answer_Description'].str.contains('Always|Yes', na=False, case=False)]

# 2. Now run the hospital-level grouping on ONLY the top-tier performance
top_hospital_stats = top_box_data.groupby('Provider_ID')['Answer_Percent'].agg(['mean', 'count']).reset_index()

# 3. Filter for reliability (at least 3-5 measures)
reliable_top_hospitals = top_hospital_stats[top_hospital_stats['count'] >= 3]

# 4. Now calculate the 5th percentile
final_floor = reliable_top_hospitals['mean'].quantile(0.05)
urgent_top_hospitals = reliable_top_hospitals[reliable_top_hospitals['mean'] <= final_floor]

print(f"Top-Box Performance Floor: {final_floor:.2f}%")
print(f"Hospitals in 'Urgent Improvement' zone: {len(urgent_top_hospitals)}")

Top-Box Performance Floor: 59.44%
Hospitals in 'Urgent Improvement' zone: 32


In [30]:
# Merge with original data to get State names
urgent_list = urgent_top_hospitals.merge(df[['Provider_ID', 'State']].drop_duplicates(), on='Provider_ID')

print("Hospitals needing urgent improvement by State:")
print(urgent_list['State'].value_counts())

Hospitals needing urgent improvement by State:
State
CA    29
AZ     2
AK     1
AL     0
AR     0
CO     0
CT     0
Name: count, dtype: int64


In [31]:
# 1. Get total reliable hospitals per state
total_per_state = reliable_top_hospitals.merge(df[['Provider_ID', 'State']].drop_duplicates(), on='Provider_ID')['State'].value_counts()

# 2. Get underperforming hospitals per state
under_per_state = urgent_list['State'].value_counts()

# 3. Calculate % of state's hospitals that are "Urgent Improvement"
failure_rate = (under_per_state / total_per_state * 100).fillna(0)

print("--- State Failure Rates (%) ---")
print(failure_rate.sort_values(ascending=False))

--- State Failure Rates (%) ---
State
CA    9.354839
AK    7.692308
AZ    2.985075
AL    0.000000
AR    0.000000
CO    0.000000
CT    0.000000
Name: count, dtype: float64


In [32]:
#-----------------------------------------------------
# 🟢 Cell 9 — Data Quality Summary (REQUIRED OUTPUT)
#-----------------------------------------------------
# --- FINAL DATA QUALITY & INTEGRITY AUDIT ---

quality_summary = {
    # 1. Processing Integrity
    "ingestion": {
        "rows_raw": len(df_raw), # Using your raw dataframe variable
        "rows_clean": len(df),
        "retention_rate": f"{(len(df) / len(df_raw) * 100):.2f}%"
    },
    
    # 2. Data Health
    "data_health": {
        "null_pct_by_feature": df.isna().mean().round(4).to_dict(),
        "duplicate_records_found": df_raw.duplicated().sum()
    },
    
    # 3. Healthcare Logic Validation (BRV)
    "business_logic": {
        "star_rating_violations": len(df[(df['Patient_Survey_Star_Rating'] < 1) | 
                                          (df['Patient_Survey_Star_Rating'] > 5)]),
        "out_of_bounds_percentages": len(df[(df['Answer_Percent'] < 0) | 
                                             (df['Answer_Percent'] > 100)])
    },
    
    # 4. Performance Insights (The "Hero" metrics)
    "clinical_insights": {
        "hospitals_analyzed": len(reliable_top_hospitals),
        "urgent_improvement_count": len(urgent_top_hospitals),
        "performance_floor_top_box": f"{final_floor:.2f}%"
    }
}
print(quality_summary)
# Display the summary in a clean format
# import json
# print(json.dumps(quality_summary, indent=4))

{'ingestion': {'rows_raw': 34999, 'rows_clean': 34999, 'retention_rate': '100.00%'}, 'data_health': {'null_pct_by_feature': {'Provider_ID': 0.0, 'Hospital_Name': 0.0, 'Address': 0.0, 'City': 0.0, 'State': 0.0, 'ZIP_Code': 0.0, 'County_Name': 0.0071, 'Phone_Number': 0.0, 'Measure_ID': 0.0, 'Question': 0.0, 'Answer_Description': 0.0, 'Patient_Survey_Star_Rating': 0.8297, 'Patient_Survey_Star_Rating_Footnote': 0.9503, 'Answer_Percent': 0.4871, 'Answer_Percent_Footnote': 0.8525, 'Linear_Mean_Value': 0.8451, 'Number_of_Completed_Surveys': 0.1157, 'Number_of_Completed_Surveys_Footnote': 0.7457, 'Survey_Response_Rate_Percent': 0.1157, 'Survey_Response_Rate_Percent_Footnote': 0.7457, 'Measure_Start_Date': 0.0, 'Measure_End_Date': 0.0, 'Location': 0.0}, 'duplicate_records_found': 0}, 'business_logic': {'star_rating_violations': 0, 'out_of_bounds_percentages': 0}, 'clinical_insights': {'hospitals_analyzed': 619, 'urgent_improvement_count': 32, 'performance_floor_top_box': '59.44%'}}


In [33]:
# Create the "Analysis-Ready" dataset
# This filters for Top-Box responses and Reliable hospital counts we found earlier
reliable_provider_ids = reliable_top_hospitals['Provider_ID']
df_analysis = df[df['Provider_ID'].isin(reliable_provider_ids)].copy()

print(f"Analysis-Ready dataset created with {df_analysis['Provider_ID'].nunique()} hospitals.")

Analysis-Ready dataset created with 619 hospitals.


In [34]:
df_analysis.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30949 entries, 0 to 34998
Data columns (total 23 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   Provider_ID                            30949 non-null  object        
 1   Hospital_Name                          30949 non-null  object        
 2   Address                                30949 non-null  object        
 3   City                                   30949 non-null  object        
 4   State                                  30949 non-null  category      
 5   ZIP_Code                               30949 non-null  object        
 6   County_Name                            30949 non-null  object        
 7   Phone_Number                           30949 non-null  object        
 8   Measure_ID                             30949 non-null  category      
 9   Question                               30949 non-null  object     

In [36]:
print(df_analysis.shape)

(30949, 23)


In [37]:
#converting clean data into new CSV File
df_analysis.to_csv('HospitalSurveyData_Cleaned.csv', index=False)